# Infield P4 or Not Models — Train V2

Split the **hitter** CSV into just the infielders (SS, 2B, 3B, 1B) to do model development.

Building out the P4 or not models for the infielders.

**Source:** `backend/ml/train/train_v2/hitters/hitters_cleaned.csv`

In [17]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

HITTER_CSV = Path("hitters_cleaned.csv")

hitters = pd.read_csv(HITTER_CSV, low_memory=False)
print(f"Loaded {len(hitters):,} rows, {len(hitters.columns)} columns")

inf = hitters[hitters["resolved_position"].isin(["SS", "2B", "3B", "1B"])]
inf.head()

Loaded 32,999 rows, 50 columns


,name,link,player_state,high_school,class,resolved_position,positions,commitment,commitment_date,height,...,of_velo_date,c_velo,c_velo_date,pop_time,pop_time_date,player_region,conference,division,college_location,committment_group
0,Koleson Abbott,https://www.prepbaseballreport.com/profiles/IN...,IN,Leo Junior/Senior,2027,1B,"1B,3B",Grace College,NaN,75.0,...,NaN,NaN,NaN,NaN,NaN,Midwest,Crossroads,NAIA,"WINONA LAKE, IN",Non D1
1,Evan Adamczewski,https://www.prepbaseballreport.com/profiles/IL...,IL,Maine South,2027,1B,"1B,OF",Northern Illinois,1/27/26,70.0,...,2/28/26,NaN,NaN,NaN,NaN,Midwest,Mid-American Conference,NCAA I,"DEKALB, IL",Non P4 D1
2,Jaxon Adcock,https://www.prepbaseballreport.com/profiles/OK...,OK,Cameron,2027,SS,"SS,RHP",Oklahoma (9/26/25),NaN,76.0,...,NaN,NaN,NaN,NaN,NaN,South,Southeastern Conference,NCAA I,"NORMAN, OK",P4
3,Tre Adcox,https://www.prepbaseballreport.com/profiles/MS...,MS,Brandon,2027,3B,"3B,RHP",New Orleans,1/29/26,73.0,...,7/26/22,NaN,NaN,NaN,NaN,South,Southland,NCAA I,"NEW ORLEANS, LA",Non P4 D1
4,Banks Addison,https://www.prepbaseballreport.com/profiles/TN...,TN,Evangelical Christian,2027,SS,"SS,C",Tennessee (9/16/25),NaN,71.0,...,NaN,NaN,NaN,NaN,NaN,South,Southeastern Conference,NCAA I,"KNOXVILLE, TN",P4


In [18]:
INF_MODELING_COLS = [
    "player_state", "resolved_position",
    "height", "weight", "throwing_hand", "hitting_handedness",
    "exit_velo_max", "exit_velo_max_date",
    "exit_velo_avg", "exit_velo_avg_date",
    "distance_max", "distance_max_date",
    "sweet_spot_p", "sweet_spot_p_date",
    "hand_speed_max", "hand_speed_max_date",
    "bat_speed_max", "bat_speed_max_date",
    "rot_acc_max", "rot_acc_max_date",
    "hard_hit_p", "hard_hit_p_date",
    "sixty_time", "sixty_time_date",
    "thirty_time", "thirty_time_date",
    "ten_yard_time", "ten_yard_time_date",
    "run_speed_max", "run_speed_max_date",
    "inf_velo", "inf_velo_date",
    "player_region", "committment_group", "commitment_date"
]

INF_MODELING_COLS = [c for c in INF_MODELING_COLS if c in inf.columns]
inf_modeling = inf[INF_MODELING_COLS]

inf_modeling = inf_modeling[inf_modeling["committment_group"] != "Unknown"]
inf_modeling = inf_modeling[inf_modeling["committment_group"] != "Non D1"]

print(inf_modeling["committment_group"].value_counts())
print(inf_modeling.shape)

committment_group
Non P4 D1    3224
P4            864
Name: count, dtype: int64
(4088, 35)


In [19]:
inf_modeling["p4_or_not"] = (inf_modeling["committment_group"] != "Non P4 D1").astype(int)

inf_modeling["p4_or_not"].value_counts()

p4_or_not
0    3224
1     864
Name: count, dtype: int64

In [20]:
# ============================================================
# STALE DATA ANALYSIS
# ============================================================
STALE_MONTHS = 24
OUTLIER_STD_DEV = 2

inf_model_recent = inf_modeling.copy()
inf_model_recent["class"] = inf.loc[inf_model_recent.index, "class"]

STAT_DATE_PAIRS = [
    ("exit_velo_max",  "exit_velo_max_date"),
    ("exit_velo_avg",  "exit_velo_avg_date"),
    ("distance_max",   "distance_max_date"),
    ("sweet_spot_p",   "sweet_spot_p_date"),
    ("hand_speed_max", "hand_speed_max_date"),
    ("bat_speed_max",  "bat_speed_max_date"),
    ("rot_acc_max",    "rot_acc_max_date"),
    ("hard_hit_p",     "hard_hit_p_date"),
    ("sixty_time",     "sixty_time_date"),
    ("thirty_time",    "thirty_time_date"),
    ("ten_yard_time",  "ten_yard_time_date"),
    ("run_speed_max",  "run_speed_max_date"),
    ("inf_velo",       "inf_velo_date"),
]
STAT_DATE_PAIRS = [(s, d) for s, d in STAT_DATE_PAIRS if s in inf_model_recent.columns and d in inf_model_recent.columns]

def parse_pbr_date(d):
    if pd.isna(d) or str(d).strip() == "":
        return pd.NaT
    try:
        return pd.to_datetime(str(d).strip(), format="%m/%d/%y")
    except Exception:
        try:
            return pd.to_datetime(str(d).strip())
        except Exception:
            return pd.NaT

inf_model_recent["grad_date"] = pd.to_datetime(inf_model_recent["class"].astype(str) + "-06-01")

for stat_col, date_col in STAT_DATE_PAIRS:
    parsed_col = f"_{date_col}_parsed"
    months_col = f"{stat_col}__months_before_grad"
    inf_model_recent[parsed_col] = inf_model_recent[date_col].apply(parse_pbr_date)
    inf_model_recent[months_col] = ((inf_model_recent["grad_date"] - inf_model_recent[parsed_col]).dt.days / 30.44).round(1)

print(f"Parsed {len(STAT_DATE_PAIRS)} stat date columns.")

print(f"{'='*70}")
print(f"STALENESS BY COMMITMENT GROUP  (threshold = {STALE_MONTHS} months)")
print(f"{'='*70}")

for group in ["P4", "Non P4 D1"]:
    mask = inf_model_recent["committment_group"] == group
    g_stale = 0
    g_total = 0
    for stat_col, _ in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        has_val  = inf_model_recent.loc[mask, stat_col].notna()
        is_stale = inf_model_recent.loc[mask, months_col] > STALE_MONTHS
        g_stale += (has_val & is_stale).sum()
        g_total += has_val.sum()
    pct = 100 * g_stale / g_total if g_total else 0
    n_players = mask.sum()
    print(f"  {group:>12}  ({n_players:>5,} players): {g_stale:>4,} / {g_total:>5,} values stale ({pct:.1f}%)")

# Outlier summary
stat_value_cols = [s for s, _ in STAT_DATE_PAIRS]
print(f"{'='*70}")
print(f"OUTLIER SUMMARY  (+-{OUTLIER_STD_DEV} std dev from group mean)")
print(f"{'='*70}")

for group in ["P4", "Non P4 D1"]:
    mask = inf_model_recent["committment_group"] == group
    tot_outliers = 0
    tot_stale_o = 0
    for stat_col in stat_value_cols:
        months_col = f"{stat_col}__months_before_grad"
        vals = inf_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std = vals.std()
        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue
        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        is_stale = inf_model_recent.loc[mask, months_col] > STALE_MONTHS
        tot_outliers += is_outlier.sum()
        tot_stale_o += (is_outlier & is_stale).sum()
    tot_fresh_o = tot_outliers - tot_stale_o
    pct = 100 * tot_stale_o / tot_outliers if tot_outliers else 0
    print(f"  {group:>12}: {tot_outliers:>4} outliers -> {tot_stale_o:>3} stale ({pct:.0f}%), {tot_fresh_o:>3} fresh ({100-pct:.0f}%)")

internal_cols = [c for c in inf_model_recent.columns if c.startswith("_")]
inf_model_recent.drop(columns=internal_cols, inplace=True)
print(f"inf_model_recent shape: {inf_model_recent.shape}")

Parsed 13 stat date columns.
STALENESS BY COMMITMENT GROUP  (threshold = 24 months)
            P4  (  864 players): 2,173 / 7,037 values stale (30.9%)
     Non P4 D1  (3,224 players): 6,751 / 26,738 values stale (25.2%)
OUTLIER SUMMARY  (+-2 std dev from group mean)
            P4:  310 outliers -> 162 stale (52%), 148 fresh (48%)
     Non P4 D1: 1061 outliers -> 482 stale (45%), 579 fresh (55%)
inf_model_recent shape: (4088, 51)


In [21]:
# ============================================================
# APPLY OPTION B — NaN only stale outliers (MULTI-PASS)
# ============================================================
MAX_PASSES = 5
grand_total = 0

for pass_i in range(1, MAX_PASSES + 1):
    pass_total = 0
    pass_log = []

    for stat_col, date_col in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        stat_nand = 0

        for group in ["P4", "Non P4 D1"]:
            mask = inf_model_recent["committment_group"] == group
            vals = inf_model_recent.loc[mask, stat_col]
            mean = vals.mean()
            std = vals.std()

            if pd.isna(mean) or pd.isna(std) or std == 0:
                continue

            is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
            is_stale = inf_model_recent.loc[mask, months_col] > STALE_MONTHS

            to_nan = is_outlier & is_stale
            n = to_nan.sum()

            if n > 0:
                inf_model_recent.loc[to_nan[to_nan].index, stat_col] = np.nan
                inf_model_recent.loc[to_nan[to_nan].index, date_col] = np.nan
                stat_nand += n

        if stat_nand > 0:
            pass_log.append({"stat": stat_col, "values_removed": stat_nand})
            pass_total += stat_nand

    grand_total += pass_total
    print(f"Pass {pass_i}: NaN'd {pass_total} stale outlier values")
    if pass_log:
        for row in pass_log:
            print(f"    {row['stat']:>15}: {row['values_removed']}")

    if pass_total == 0:
        print(f"  No more stale outliers found — stopping.")
        break

print(f"Total across {pass_i} passes: {grand_total} values NaN'd")

# Verify
remaining = 0
for stat_col, _ in STAT_DATE_PAIRS:
    months_col = f"{stat_col}__months_before_grad"
    for group in ["P4", "Non P4 D1"]:
        mask = inf_model_recent["committment_group"] == group
        vals = inf_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std = vals.std()
        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue
        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        is_stale = inf_model_recent.loc[mask, months_col] > STALE_MONTHS
        remaining += (is_outlier & is_stale).sum()

print(f"Stale outliers remaining: {remaining}")
print(f"inf_model_recent shape: {inf_model_recent.shape}")

Pass 1: NaN'd 644 stale outlier values
      exit_velo_max: 100
      exit_velo_avg: 56
       distance_max: 67
       sweet_spot_p: 40
     hand_speed_max: 46
      bat_speed_max: 64
        rot_acc_max: 34
         hard_hit_p: 4
         sixty_time: 79
        thirty_time: 24
      ten_yard_time: 15
      run_speed_max: 30
           inf_velo: 85
Pass 2: NaN'd 257 stale outlier values
      exit_velo_max: 51
      exit_velo_avg: 18
       distance_max: 23
       sweet_spot_p: 45
     hand_speed_max: 11
      bat_speed_max: 11
        rot_acc_max: 7
         hard_hit_p: 5
         sixty_time: 42
        thirty_time: 7
      ten_yard_time: 9
      run_speed_max: 8
           inf_velo: 20
Pass 3: NaN'd 41 stale outlier values
      exit_velo_max: 6
       distance_max: 2
     hand_speed_max: 1
      bat_speed_max: 2
         hard_hit_p: 4
         sixty_time: 20
           inf_velo: 6
Pass 4: NaN'd 11 stale outlier values
      exit_velo_max: 3
         hard_hit_p: 1
         sixty_time

## Logistic Regression Baseline

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

FEATURES = [
    "height", "weight",
    "exit_velo_max", "distance_max", "bat_speed_max",
    "sixty_time", "inf_velo",
]

X = inf_model_recent[FEATURES]
y = inf_model_recent["p4_or_not"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ("impute", KNNImputer(n_neighbors=10)),
    ("scale", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["Non P4 D1", "P4"]))
print(confusion_matrix(y_test, y_pred), "\n")

coefs = pd.Series(pipe["lr"].coef_[0], index=FEATURES).sort_values()
print(coefs)

              precision    recall  f1-score   support

   Non P4 D1       0.85      0.60      0.71       645
          P4       0.29      0.61      0.40       173

    accuracy                           0.61       818
   macro avg       0.57      0.61      0.55       818
weighted avg       0.74      0.61      0.64       818

[[390 255]
 [ 67 106]] 

sixty_time      -0.388945
bat_speed_max   -0.015739
exit_velo_max    0.130516
height           0.153683
weight           0.169741
inf_velo         0.179690
distance_max     0.190804
dtype: float64


## Generate Out-of-Fold D1 Probabilities

We need  as a meta-feature for the P4 model. Can't use the saved D1 model directly because these D1 players were in its training data — that would leak. Instead, retrain the D1 model on the **full infielder dataset** (including Non-D1) with cross_val_predict to get leak-free calibrated probabilities for every player, then filter to just the D1 subset.

In [23]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
import numpy as np

# Load the FULL infielder dataset (including Non-D1) for D1 model training
hitters_full = pd.read_csv(Path("hitters_cleaned.csv"), low_memory=False)
inf_full = hitters_full[hitters_full["resolved_position"].isin(["SS", "2B", "3B", "1B"])].copy()
inf_full = inf_full[inf_full["committment_group"] != "Unknown"]
inf_full["d1_or_not"] = (inf_full["committment_group"] != "Non D1").astype(int)

FEATURES_D1 = ["height", "weight", "exit_velo_max", "distance_max", "sixty_time", "inf_velo", "bat_speed_max"]

X_full = inf_full[FEATURES_D1]
y_full = inf_full["d1_or_not"]
neg, pos = (y_full == 0).sum(), (y_full == 1).sum()

print(f"Full INF dataset: {len(inf_full)} players ({pos} D1, {neg} Non-D1)")

xgb_d1 = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.01,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=3.0, min_child_weight=5,
    scale_pos_weight=neg / pos, eval_metric="logloss",
    random_state=42, enable_categorical=False,
)
cal_d1 = CalibratedClassifierCV(xgb_d1, method="isotonic", cv=3)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = cross_val_predict(cal_d1, X_full, y_full, cv=skf, method="predict_proba")[:, 1]

inf_full["d1_prob"] = oof_probs

# Filter to D1 players only and merge into inf_model_recent
d1_players = inf_full[inf_full["d1_or_not"] == 1][["d1_prob"]]
inf_model_recent = inf_model_recent.join(d1_players, how="left")

print(f"D1 prob stats for D1 players (n={inf_model_recent['d1_prob'].notna().sum()}):")
print(f"  mean: {inf_model_recent['d1_prob'].mean():.3f}")
print(f"  std:  {inf_model_recent['d1_prob'].std():.3f}")
print(f"  min:  {inf_model_recent['d1_prob'].min():.3f}")
print(f"  max:  {inf_model_recent['d1_prob'].max():.3f}")

print(f"D1 prob by group:")
for group in ["P4", "Non P4 D1"]:
    mask = inf_model_recent["committment_group"] == group
    vals = inf_model_recent.loc[mask, "d1_prob"]
    print(f"  {group:>12}: mean={vals.mean():.3f}  std={vals.std():.3f}  median={vals.median():.3f}")

Full INF dataset: 15451 players (4088 D1, 11363 Non-D1)
D1 prob stats for D1 players (n=4088):
  mean: 0.386
  std:  0.199
  min:  0.003
  max:  1.000
D1 prob by group:
            P4: mean=0.479  std=0.219  median=0.438
     Non P4 D1: mean=0.362  std=0.186  median=0.341


## XGBoost — P4 vs Non-P4 D1 (7 core + d1_prob)

Using the 7 production features plus the D1 probability as a meta-feature.

In [27]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

FEATURES_P4 = ["height", "weight", "exit_velo_max", "distance_max", "sixty_time", "inf_velo", "bat_speed_max", "d1_prob"]

X = inf_model_recent[FEATURES_P4]
y = inf_model_recent["p4_or_not"]
neg, pos = (y == 0).sum(), (y == 1).sum()
print(f"Class balance: {neg} Non-P4 D1, {pos} P4  (ratio {neg/pos:.2f}:1)")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_p4 = XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.5, reg_lambda=5.0, min_child_weight=8,
    scale_pos_weight=neg / pos, eval_metric="logloss",
    random_state=42, enable_categorical=False,
)

xgb_p4.fit(X_train, y_train)
y_pred = xgb_p4.predict(X_test)
y_proba = xgb_p4.predict_proba(X_test)[:, 1]

print(f"XGBoost — {len(FEATURES_P4)} Features (7 core + d1_prob)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["Non P4 D1", "P4"]))
print(confusion_matrix(y_test, y_pred))

auc = roc_auc_score(y_test, y_proba)
print(f"AUC-ROC: {auc:.4f}")

importances = pd.Series(xgb_p4.feature_importances_, index=FEATURES_P4).sort_values(ascending=False)
print(f"Feature importance:")
print(importances.to_string())

# Stratified 5-Fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_validate(
    xgb_p4, X, y, cv=skf,
    scoring=["roc_auc", "f1", "precision", "recall", "accuracy"],
    return_train_score=True,
)

print(f"Stratified 5-Fold CV:")
print("=" * 60)
for metric in ["roc_auc", "accuracy", "f1", "precision", "recall"]:
    tr = cv[f"train_{metric}"]
    te = cv[f"test_{metric}"]
    label = "AUC-ROC" if metric == "roc_auc" else metric
    print(f"  {label:>12}:  train {tr.mean():.3f} (+/- {tr.std():.3f})  |  test {te.mean():.3f} (+/- {te.std():.3f})")
print(f"Overfit gap (AUC): {cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean():.3f}")

# Calibration
xgb_p4_cal = CalibratedClassifierCV(xgb_p4, method="isotonic", cv=5)
xgb_p4_cal.fit(X_train, y_train)
y_proba_cal = xgb_p4_cal.predict_proba(X_test)[:, 1]
y_pred_cal = (y_proba_cal >= 0.3).astype(int)

fraction_raw, mean_raw = calibration_curve(y_test, y_proba, n_bins=8)
fraction_cal, mean_cal = calibration_curve(y_test, y_proba_cal, n_bins=8)

brier_raw = brier_score_loss(y_test, y_proba)
brier_cal = brier_score_loss(y_test, y_proba_cal)

print(f"Calibrated model (threshold=0.3):")
print("=" * 60)
print(classification_report(y_test, y_pred_cal, target_names=["Non P4 D1", "P4"]))
print(confusion_matrix(y_test, y_pred_cal))

print(f"Brier:  raw = {brier_raw:.4f}  |  calibrated = {brier_cal:.4f}  |  improvement = {brier_raw - brier_cal:.4f}")

print(f"Calibration comparison (raw vs calibrated):")
print("=" * 75)
print(f"  {'--- RAW ---':^28}   |   {'--- CALIBRATED ---':^28}")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}   |   {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
print(f"  {'-'*28}   |   {'-'*28}")

max_rows = max(len(mean_raw), len(mean_cal))
for i in range(max_rows):
    if i < len(mean_raw):
        r_gap = mean_raw[i] - fraction_raw[i]
        raw_str = f"  {mean_raw[i]:>10.2f}  {fraction_raw[i]:>8.2f}  {r_gap:>+8.2f}"
    else:
        raw_str = f"  {'':>28}"
    if i < len(mean_cal):
        c_gap = mean_cal[i] - fraction_cal[i]
        cal_str = f"{mean_cal[i]:>10.2f}  {fraction_cal[i]:>8.2f}  {c_gap:>+8.2f}"
        flag = "  <- off" if abs(c_gap) > 0.10 else ""
        cal_str += flag
    else:
        cal_str = f"{'':>28}"
    print(f"{raw_str}   |   {cal_str}")

auc_cal = roc_auc_score(y_test, y_proba_cal)
max_gap_cal = max(abs(mean_cal[i] - fraction_cal[i]) for i in range(len(mean_cal)))
print(f"AUC-ROC (calibrated): {auc_cal:.4f}")
print(f"Max calibration gap: {max_gap_cal:.2f}")

# Comparison
print(f"{'=' * 60}")
print("COMPARISON: LogReg baseline vs XGBoost + d1_prob")
print(f"{'=' * 60}")
rpt = classification_report(y_test, y_pred, target_names=["Non P4 D1", "P4"], output_dict=True)
print(f"  LogReg (7 feat):       P4 F1=0.42  acc=0.61")
print(f"  XGBoost (8 feat):      P4 F1={rpt['P4']['f1-score']:.2f}  acc={rpt['accuracy']:.2f}  AUC={auc:.3f}  Brier(cal)={brier_cal:.4f}")

Class balance: 3224 Non-P4 D1, 864 P4  (ratio 3.73:1)
XGBoost — 8 Features (7 core + d1_prob)
              precision    recall  f1-score   support

   Non P4 D1       0.87      0.67      0.75       645
          P4       0.33      0.62      0.43       173

    accuracy                           0.66       818
   macro avg       0.60      0.64      0.59       818
weighted avg       0.75      0.66      0.69       818

[[431 214]
 [ 66 107]]
AUC-ROC: 0.6911
Feature importance:
d1_prob          0.247467
sixty_time       0.125078
exit_velo_max    0.115910
distance_max     0.108895
height           0.107919
inf_velo         0.104363
weight           0.096509
bat_speed_max    0.093859
Stratified 5-Fold CV:
       AUC-ROC:  train 0.795 (+/- 0.004)  |  test 0.700 (+/- 0.017)
      accuracy:  train 0.701 (+/- 0.005)  |  test 0.656 (+/- 0.008)
            f1:  train 0.506 (+/- 0.004)  |  test 0.430 (+/- 0.012)
     precision:  train 0.388 (+/- 0.004)  |  test 0.331 (+/- 0.006)
        recall:  t

In [28]:
import joblib
import json
import os
from datetime import datetime

# ============================================================
# Save calibrated XGBoost P4 model to production model directory
# ============================================================
THRESHOLD = 0.35
VERSION = f"version_{datetime.now().strftime('%m%d%Y')}"
SAVE_DIR = os.path.abspath(os.path.join(
    os.getcwd(), "..", "..", "..", "models", "models_inf", "models_p4_or_not_inf", VERSION
))
os.makedirs(SAVE_DIR, exist_ok=True)

FEATURES_P4 = ["height", "weight", "exit_velo_max", "distance_max", "sixty_time", "inf_velo", "bat_speed_max", "d1_prob"]

X = inf_model_recent[FEATURES_P4]
y = inf_model_recent["p4_or_not"]
neg, pos = (y == 0).sum(), (y == 1).sum()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_final = XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.5, reg_lambda=5.0, min_child_weight=8,
    scale_pos_weight=neg / pos, eval_metric="logloss",
    random_state=42, enable_categorical=False,
)
xgb_final.fit(X_train, y_train)

xgb_final_cal = CalibratedClassifierCV(xgb_final, method="isotonic", cv=5)
xgb_final_cal.fit(X_train, y_train)

# --- Eval metrics on holdout ---
y_proba_test = xgb_final_cal.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= THRESHOLD).astype(int)

auc = roc_auc_score(y_test, y_proba_test)
brier = brier_score_loss(y_test, y_proba_test)
report = classification_report(y_test, y_pred_test, target_names=["Non P4 D1", "P4"], output_dict=True)
fraction_cal, mean_cal = calibration_curve(y_test, y_proba_test, n_bins=8)
max_cal_gap = max(abs(m - f) for m, f in zip(mean_cal, fraction_cal))

# --- Save model ---
joblib.dump(xgb_final_cal, os.path.join(SAVE_DIR, "calibrated_xgb_model.pkl"))
print(f"Saved: calibrated_xgb_model.pkl")

# --- Save model config ---
model_config = {
    "model_version": f"inf_p4_xgb_cal_{VERSION}",
    "creation_date": datetime.now().isoformat(),
    "model_type": "calibrated_xgboost_infielder_p4",
    "calibration_method": "isotonic",
    "threshold": THRESHOLD,
    "features": FEATURES_P4,
    "requires_d1_prob": True,
    "hyperparameters": {
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 1.5,
        "reg_lambda": 5.0,
        "min_child_weight": 8,
        "scale_pos_weight": round(neg / pos, 4),
    },
    "performance_metrics": {
        "auc_roc": round(auc, 4),
        "brier_score": round(brier, 4),
        "max_calibration_gap": round(max_cal_gap, 4),
        "p4_precision": round(report["P4"]["precision"], 4),
        "p4_recall": round(report["P4"]["recall"], 4),
        "p4_f1": round(report["P4"]["f1-score"], 4),
        "accuracy": round(report["accuracy"], 4),
        "threshold_used": THRESHOLD,
    },
    "dataset_info": {
        "total_samples": len(inf_model_recent),
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        "p4_rate": round(pos / (neg + pos), 4),
        "d1_only": True,
        "stale_cleaning": "Option B multi-pass (stale outliers only, +/-2 std + >24mo)",
    },
    "calibration_curve": {
        "predicted": [round(float(m), 4) for m in mean_cal],
        "actual": [round(float(f), 4) for f in fraction_cal],
    },
}

with open(os.path.join(SAVE_DIR, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)
print(f"Saved: model_config.json")

# --- Save feature metadata ---
feature_metadata = {
    "features": FEATURES_P4,
    "required_columns": ["height", "weight", "exit_velo_max", "distance_max", "sixty_time", "inf_velo", "bat_speed_max"],
    "meta_features": ["d1_prob"],
    "notes": "d1_prob comes from the calibrated D1 infielder model. XGBoost handles NaN natively. No imputation or scaling needed.",
}

with open(os.path.join(SAVE_DIR, "feature_metadata.json"), "w") as f:
    json.dump(feature_metadata, f, indent=2)
print(f"Saved: feature_metadata.json")

# --- Print summary ---
print(f"\nModel saved to: {SAVE_DIR}")
print(f"\n{'=' * 60}")
print(f"PRODUCTION MODEL METRICS (threshold={THRESHOLD})")
print(f"{'=' * 60}")
print(f"  AUC-ROC:           {auc:.4f}")
print(f"  Brier score:       {brier:.4f}")
print(f"  Max cal gap:       {max_cal_gap:.4f}")
print(f"  P4 Precision:      {report['P4']['precision']:.4f}")
print(f"  P4 Recall:         {report['P4']['recall']:.4f}")
print(f"  P4 F1:             {report['P4']['f1-score']:.4f}")
print(f"  Accuracy:          {report['accuracy']:.4f}")
print(f"  Features:          {len(FEATURES_P4)} ({', '.join(FEATURES_P4)})")
print(f"  Threshold:         {THRESHOLD}")
print(f"  Training samples:  {len(X_train)}")
print(f"  Test samples:      {len(X_test)}")
print(f"  P4 base rate:      {pos/(neg+pos):.1%}")

print(f"\nCalibration curve:")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
for pred, actual in zip(mean_cal, fraction_cal):
    gap = pred - actual
    flag = "  <- off" if abs(gap) > 0.10 else ""
    print(f"  {pred:>10.2f}  {actual:>8.2f}  {gap:>+8.2f}{flag}")

print(f"\nFiles in {VERSION}/:")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:<35} {size:>8,} bytes")

Saved: calibrated_xgb_model.pkl
Saved: model_config.json
Saved: feature_metadata.json

Model saved to: /Users/ryankolodziejczyk/Documents/AI Baseball Recruitment/code/backend/ml/models/models_inf/models_p4_or_not_inf/version_04212026

PRODUCTION MODEL METRICS (threshold=0.35)
  AUC-ROC:           0.6921
  Brier score:       0.1504
  Max cal gap:       0.3599
  P4 Precision:      0.4417
  P4 Recall:         0.3064
  P4 F1:             0.3618
  Accuracy:          0.7714
  Features:          8 (height, weight, exit_velo_max, distance_max, sixty_time, inf_velo, bat_speed_max, d1_prob)
  Threshold:         0.35
  Training samples:  3270
  Test samples:      818
  P4 base rate:      21.1%

Calibration curve:
   Predicted    Actual       Gap
        0.07      0.10     -0.03
        0.19      0.18     +0.01
        0.30      0.30     +0.00
        0.42      0.36     +0.06
        0.56      0.69     -0.13  <- off
        0.64      1.00     -0.36  <- off

Files in version_04212026/:
  calibrated